# Step 1: Genesis で Go2（四足ロボット）を歩かせる
Colab のランタイムを **GPU**（ランタイム → ランタイムのタイプを変更）にしてから、上のセルから順に実行してください。
エラーが出たら、そのセルのログを貼ってください。

In [ ]:
!nvidia-smi

## 1. インストール

In [ ]:
!pip install -q genesis-world "rsl-rl-lib>=5.0.0" tensorboard moviepy imageio imageio-ffmpeg
!git clone -q https://github.com/Genesis-Embodied-AI/Genesis.git
import torch, genesis as gs
print(torch.__version__, torch.cuda.is_available())

## 2. 学習設定の変更
標準サンプルは「前に 0.5 m/s」だけを学習します。あとで VLM が向きを変えられるように、前進・横移動・旋回を広めに学習させます。

In [ ]:
import re, pathlib
p = pathlib.Path("Genesis/examples/locomotion/go2_train.py")
s = p.read_text()
s = s.replace('"lin_vel_x_range": [0.5, 0.5]', '"lin_vel_x_range": [0.0, 0.8]')
s = s.replace('"lin_vel_y_range": [0, 0]', '"lin_vel_y_range": [0.0, 0.0]')
s = s.replace('"ang_vel_range": [0, 0]', '"ang_vel_range": [-1.0, 1.0]')
p.write_text(s)
print(re.findall(r'"(?:lin_vel_x_range|lin_vel_y_range|ang_vel_range)".*', s))

# 学習には描画が要らない。Colab では描画(EGL)の初期化で落ちるため、Rasterizer の build を無効化する
s = p.read_text()
patch = (
    "import genesis.vis.rasterizer as _gs_rast\n"
    "_gs_rast.Rasterizer.build = lambda self: None  # Colab: skip offscreen EGL init (training only)\n"
)
if "_gs_rast" not in s:
    key = "import genesis as gs\n"
    assert key in s, "go2_train.py の構成が想定と違います"
    s = s.replace(key, key + patch, 1)
    p.write_text(s)
print("rasterizer patched:", "_gs_rast" in p.read_text())


## 3. 学習（GPU で数分〜十数分）

In [ ]:
%cd Genesis/examples/locomotion
!python go2_train.py -e go2-walking --max-iterations 301

## 4. 学習結果をまとめてダウンロード
Colab では描画が動かないため、**動画の作成は手元の PC** で行います。
学習済みモデルと設定、環境ファイルを zip にして、ダウンロードします。

In [ ]:
import shutil, os
os.makedirs("/content/export", exist_ok=True)
shutil.copy("go2_env.py", "/content/export/go2_env.py")
shutil.copytree("logs/go2-walking", "/content/export/logs/go2-walking", dirs_exist_ok=True)
# 一番新しいモデルだけ残す
ms = sorted([f for f in os.listdir("/content/export/logs/go2-walking") if f.startswith("model_")],
            key=lambda f: int(f.split("_")[1].split(".")[0]))
for f in ms[:-1]:
    os.remove("/content/export/logs/go2-walking/" + f)
print("残すモデル:", ms[-1])
shutil.make_archive("/content/go2_export", "zip", "/content/export")
from google.colab import files
files.download("/content/go2_export.zip")